In [ ]:
# Run once to make sure everything is installed
# You can skip this cell if your environment is already set up
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'requests', 'pandas', 'nltk'])

In [ ]:
import requests
import json
import re
import time
import pandas as pd

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer


In [ ]:
# ---------------Configuration

# Steam App IDs for popular games (change or add more as needed)
# Find an app ID by visiting a store page: store.steampowered.com/app/{appid}
GAME_IDS = {
    730:   'Counter-Strike 2',
    570:   'Dota 2',
    1091500: 'Cyberpunk 2077',
}

MAX_REVIEWS_PER_GAME = 500
REVIEWS_PER_REQUEST  = 100
DELAY_BETWEEN_CALLS  = 1.0   # seconds - be polite to the API
OUTPUT_CSV           = 'steam_reviews_raw.csv'

In [ ]:
def fetch_reviews_for_game(appid: int, game_name: str, max_reviews: int = 500) -> list[dict]:
    """
    Fetches up to `max_reviews` reviews for a Steam game using cursor-based pagination.

    Returns a list of dicts, each containing:
        review_text, voted_up, timestamp_created, appid, game_name
    """
    base_url = f'https://store.steampowered.com/appreviews/{appid}'
    params = {
        'json':           1,
        'language':       'english',
        'review_type':    'all',        # 'positive', 'negative', 'all'
        'purchase_type':  'all',
        'num_per_page':   REVIEWS_PER_REQUEST,
        'cursor':         '*',          # start cursor - Steam uses '*' for first page
        'filter':         'recent',     # 'recent','updated', 'all'
    }

    collected = []
    seen_cursors = set()

    while len(collected) < max_reviews:
        try:
            response = requests.get(base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
        except requests.RequestException as e:
            print(f'   Request error for appid {appid}: {e}')
            break

        if data.get('success') != 1:
            print(f'   API returned success=0 for appid {appid}')
            break

        reviews = data.get('reviews', [])
        if not reviews:
            print(f'   No more reviews available for {game_name}.')
            break

        for r in reviews:
            collected.append({
                'review_text':       r.get('review', ''),
                'voted_up':          r.get('voted_up', None),       # True = recommended
                'timestamp_created': r.get('timestamp_created', None),
                'appid':             appid,
                'game_name':         game_name,
            })

        new_cursor = data.get('cursor', '')
        print(f'  Fetched {len(collected):>4} reviews so far for {game_name}...')

        # Stop if cursor hasn't changed (end of results)
        if new_cursor in seen_cursors or new_cursor == params['cursor']:
            print(f'  [✓] Cursor loop detected - stopping for {game_name}.')
            break

        seen_cursors.add(new_cursor)
        params['cursor'] = new_cursor
        time.sleep(DELAY_BETWEEN_CALLS)

    return collected[:max_reviews]

In [ ]:
# --------------- Collect reviews for all configured games
all_reviews = []

for appid, name in GAME_IDS.items():
    print(f'\nFetching reviews for: {name} (appid={appid})')
    reviews = fetch_reviews_for_game(appid, name, max_reviews=MAX_REVIEWS_PER_GAME)
    all_reviews.extend(reviews)
    print(f'  Done. Total reviews collected so far: {len(all_reviews)}')

print(f'\nTotal reviews collected: {len(all_reviews)}')

In [ ]:
# ------------- Convert to DataFrame and inspect
df_raw = pd.DataFrame(all_reviews)

# Convert Unix timestamp to a readable datetime
df_raw['date'] = pd.to_datetime(df_raw['timestamp_created'], unit='s', errors='coerce')

print(f'Shape: {df_raw.shape}')
print(f'\nSentiment distribution:')
print(df_raw['voted_up'].value_counts().rename({True: 'Positive reviews', False: 'Negative reviews'}).to_string())
print(f'\nReviews per game:')
print(df_raw['game_name'].value_counts().to_string())

df_raw.head(3)

In [ ]:
# ----------- Drop rows with empty review text
before = len(df_raw)
df_raw = df_raw[df_raw['review_text'].str.strip().str.len() > 0].reset_index(drop=True)
print(f'Removed {before - len(df_raw)} empty reviews. Remaining: {len(df_raw)}')

In [ ]:
# ----------- Save raw dataset
df_raw.to_csv(OUTPUT_CSV, index=False)
print(f'Raw dataset saved to: {OUTPUT_CSV}')

In [ ]:
#  Load from CSV (skip this cell if df_raw is already in memory)
# df_raw = pd.read_csv('steam_reviews_raw.csv')
# df_raw['date'] = pd.to_datetime(df_raw['date'], errors='coerce')
# print(f'Loaded {len(df_raw)} reviews from CSV.')

In [ ]:
# ------------- Stopwords list
STOP_WORDS = set(stopwords.words('english'))

# Optional: add game-specific noise words that add no signal
CUSTOM_STOPWORDS = {'game', 'games', 'play', 'played', 'player', 'steam',
                    'get', 'got', 'one', 'would', 'really', 'also', 'like'}
STOP_WORDS.update(CUSTOM_STOPWORDS)

print(f'Total stopwords: {len(STOP_WORDS)}')

In [ ]:
def preprocess_review(text: str) -> str:
    """
    Clean a raw review string and return a preprocessed string of tokens.
    Steps:
        1. Lowercase
        2. Strip URLs
        3. Remove non-ASCII characters
        4. Remove punctuation and digits
        5. Tokenize
        6. Remove stopwords and short tokens
        7. Join into a single clean string
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    #Lowercase
    text = text.lower()

    #Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    #Remove non-ASCII (emoji, foreign chars)
    text = text.encode('ascii', errors='ignore').decode()

    #Remove punctuation and digits -keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Tokenize
    tokens = word_tokenize(text)

    #Remove stopwords and very short tokens
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]

    # 7. Rejoin
    return ' '.join(tokens)

In [ ]:
# --------------- Quick sanity check
sample = "10/10 would NOT recommend!!!!! The game crashed 5 times. https://store.steam.com"
print('Before:', sample)
print('After: ', preprocess_review(sample))

In [ ]:
#--------------- Apply preprocessing to entire dataset
print('Preprocessing reviews....(this may take time)')

df_raw['review_clean'] = df_raw['review_text'].apply(preprocess_review)

# Drop rows where preprocessing produced an empty string
before = len(df_raw)
df_clean = df_raw[df_raw['review_clean'].str.strip().str.len() > 0].reset_index(drop=True)
print(f'Dropped {before - len(df_clean)} empty rows after preprocessing.')
print(f'Final dataset shape: {df_clean.shape}')

In [ ]:
# --------------- Preview some examples
pd.set_option('display.max_colwidth', 120)
df_clean[['review_text', 'review_clean', 'voted_up']].head(5)

In [ ]:
#-------------- Basic token statistics
df_clean['token_count'] = df_clean['review_clean'].str.split().str.len()

print('Token count statistics:')
print(df_clean['token_count'].describe().round(2).to_string())
print(f'\nReviews with < 5 tokens: {(df_clean["token_count"] < 5).sum()}')

In [ ]:
# --------------- Optional: filter out very short reviews
MIN_TOKENS = 5
before = len(df_clean)
df_clean = df_clean[df_clean['token_count'] >= MIN_TOKENS].reset_index(drop=True)
print(f'Kept {len(df_clean)} reviews with >= {MIN_TOKENS} tokens (removed {before - len(df_clean)}).')

In [ ]:
# --------------- Save preprocessed dataset
CLEAN_CSV = 'steam_reviews_clean.csv'
df_clean.to_csv(CLEAN_CSV, index=False)
print(f'Preprocessed dataset saved to: {CLEAN_CSV}')
print(f'Columns: {list(df_clean.columns)}')

In [ ]:
from collections import Counter
import numpy as np

# -----------------------Calculate TF-IDF
# Load preprocessed reviews
df = pd.read_csv('steam_reviews_clean.csv')
valid_not_null = df['review_clean'].notna()
valid_not_empty = df['review_clean'].str.strip() != ''
df = df[valid_not_null & valid_not_empty].reset_index(drop=True)
corpus = [review.split() for review in df['review_clean']]
print(f'Documents : {len(corpus)}')
print(f'Example   : {corpus[0][:10]}')

# --- Build vocab
def build_vocabulary(corpus: list[list[str]]) -> dict[str, int]:
    """Maps each unique token to a column index (sorted for reproducibility)."""
    all_tokens = set()
    for doc in corpus:
        for token in doc:
            all_tokens.add(token)
    return {token: idx for idx, token in enumerate(sorted(all_tokens))}

vocab        = build_vocabulary(corpus)
idx_to_token = {v: k for k, v in vocab.items()}

print(f'Vocabulary size : {len(vocab):,} unique tokens')

# -- Compute TF matrix
def compute_tf_matrix(corpus: list[list[str]], vocab: dict[str, int]) -> np.ndarray:
    """Each cell [i, j] = raw count of vocab token j in document i."""
    tf = np.zeros((len(corpus), len(vocab)), dtype=np.float32)
    for doc_idx, doc in enumerate(corpus):
        for token, count in Counter(doc).items():
            if token in vocab:
                tf[doc_idx, vocab[token]] = count
    return tf


tf_matrix = compute_tf_matrix(corpus, vocab)
print(f'TF matrix shape : {tf_matrix.shape}')

# -- Compute IDF vector
# IDF(t) = log(N / df(t))
def compute_idf_vector(tf_matrix: np.ndarray) -> np.ndarray:
    """Standard IDF: log(N / df). Terms with df=0 get IDF=0."""
    N  = tf_matrix.shape[0]
    df = np.count_nonzero(tf_matrix, axis=0)
    return np.where(df > 0, np.log(N / df), 0.0).astype(np.float32)


idf_vector = compute_idf_vector(tf_matrix)

# Inspect: lowest and highest IDF terms
idx_sorted = np.argsort(idf_vector)
print('10 most common terms (lowest IDF):')
for i in idx_sorted[:10]:
    print(f'  {idx_to_token[i]:<20} idf = {idf_vector[i]:.4f}')
print('\n10 rarest terms (highest IDF):')
for i in idx_sorted[-10:][::-1]:
    print(f'  {idx_to_token[i]:<20} idf = {idf_vector[i]:.4f}')

# Compute TF- IDF and apply L2 normalization

def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    """
    L2-normalizes each row so all vectors have unit length.
    Rows with zero norm (all-zero vectors) stay as zero - no division.
    """
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)  # shape (n_docs, 1)
    return np.where(norms > 0, matrix / norms, 0.0).astype(np.float32)


tfidf_matrix      = tf_matrix * idf_vector          # TF × IDF  (n_docs, vocab)
tfidf_matrix_l2   = l2_normalize(tfidf_matrix)      # L2 normalize each row

# Sanity check: all row norms should be ~1.0 (or 0 for empty docs)
row_norms = np.linalg.norm(tfidf_matrix_l2, axis=1)
print(f'TF-IDF L2 matrix shape : {tfidf_matrix_l2.shape}')
print(f'Row norm min/max       : {row_norms.min():.4f} / {row_norms.max():.4f}  (expect 0 or 1.0)')

# Save output
np.save('tfidf_matrix_l2.npy', tfidf_matrix_l2)
np.save('idf_vector.npy',      idf_vector)
pd.DataFrame(vocab.items(), columns=['token', 'index']).to_csv('vocabulary.csv', index=False)

print('Saved:')
print(f'  tfidf_matrix_l2.npy  {tfidf_matrix_l2.shape}')
print(f'  idf_vector.npy       {idf_vector.shape}')
print(f'  vocabulary.csv       {len(vocab)} tokens')

In [ ]:
# --------------------- Doing sentiment analysis through VADER
#download and intialize VADEr lexicon
nltk.download('vader_lexicon', quiet=True)

sia= SentimentIntensityAnalyzer()

# applying VADER to raw text reviews first
print ("computing VADER sentiment scores for raw reviews...")
df_clean ['vader_scores'] = df_clean['review_text'].apply(lambda x: sia.polarity_scores(x))

#get compound score 
df_clean['vader_compound'] = df_clean['vader_scores'].apply(lambda x: x['compound'])

# classify as positive, and negative based on the compound score from before (>=.05 positive, <=-.05 negative)
df_clean["vader_sentiment"] = df_clean['vader_compound'].apply(
    lambda x: 'positive' if x >= 0.05 else ('negative' if x <= -0.05 else 'neutral')
)

print(f'VADER sentiment distribution:\n {df_clean["vader_sentiment"].value_counts().to_string()}')

print(f'VADER vs steam Sentiment analysis labels')
comparison = pd.crosstab(df_clean['voted_up'], df_clean['vader_sentiment'], rownames=['Steam Sentiment'], colnames=['VADER Sentiment'])
print(comparison.to_string())






In [ ]:
# Setup for Logiscitic regression with TF-IDF features
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split  

#loading the TF-IDF features and labels
tfidf_matrix_12 = np.load('tfidf_matrix_l2.npy')
X = tfidf_matrix_12
y = df_clean['voted_up'].values.astype(int)  # converts boolean to int   

#data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'training set: {X_train.shape[0]}, reviews')
print(f"test set: {X_test.shape[0]}, reviews")
print (f'features: {X_train.shape[1]}')

In [ ]:
# Logistic Regression model
class LogisticRegressionModel:
    def __init__ (self, LRate=0.01, max_iter=1000):
        self.LRate = LRate
        self.max_iter = max_iter
        self.weights= None
        self.bias = None
        self.costs =[]

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def calculate_cost (self, theta, X, y):
        m = len(y)
        h_theta = self.sigmoid(X.dot(theta))
        term1 = np.dot(y.T, np.log(h_theta))
        term2 = np.dot((1 - y).T, np.log(1 - h_theta + 1e-15))
        J = -1 / m* (term1 + term2)
        return J
    
    def calculate_gradient(self, theta, X, y):
        m = len(y)
        h_theta = self.sigmoid(X.dot(theta))
        grad = 1 / m * X.T.dot(h_theta - y)
        return grad
    
    def fit(self, X, y):
        #compute gradiuents and update weights
        X = np.concatenate([np.ones((X.shape[0], 1)), X], axis=1)  #add bias
        
        self.theta = np.zeros(X.shape[1])  # Initialize theta
        
        print(f"initial cost: {self.calculate_cost(self.theta, X, y):.6f}")
        
        # Gradient descent
        for i in range(self.max_iter):
            grad = self.calculate_gradient(self.theta, X, y)
            self.theta -= self.LRate * grad    
            cost = self.calculate_cost(self.theta, X, y)
            self.costs.append(cost)
            
            if i % 100 == 0:
                print(f"iteration {i+1}/ {self.max_iter}, cost: {cost:.6f}")
                
    def predict_proba(self, X):
        "predicting probabulities"
        X = np.concatenate([np.ones((X.shape[0], 1)), X], axis=1)
        return self.sigmoid(X.dot(self.theta))
    
    def predict(self, X, threshold=0.5):
        "predicting binary labels based on threshold"
        return (self.predict_proba(X) >= threshold).astype(int)
        

In [ ]:
# train and evaluate the model

lr_model = LogisticRegressionModel(LRate=0.1, max_iter=500)
lr_model.fit(X_train, y_train)

#prediuctions
y_pred = lr_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'Logistic Regression Accuracy: {accuracy:.4f}')
print (f"classification report:\n {classification_report(y_test, y_pred, target_names=['Negative', 'Positive'])}")

In [ ]:
#plotting training cost over iterations
plt.figure(figsize=(10, 5))
plt.plot(lr_model.costs, linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.title('Training Cost over Iterations')
plt.grid(True, alpha=0.2)
plt.show()